In [1]:
import pandas as pd
o = pd.read_csv("oracle_orders.csv")
f = pd.read_csv("adaptive_forecast.csv")
a = pd.read_csv("salesforce_accounts.csv")
p = pd.read_csv("product_master.csv")

for name, df in [("orders", o), ("forecast", f), ("accounts", a), ("products", p)]:
    print("=====", name, df.shape)
    print(df.head())
    print(df.isna().sum())

print("duplicated order_id:", o.order_id.duplicated().sum())
print(o.describe())
print(o.order_status.value_counts())

===== orders (128, 10)
   order_id customer_id product_id  order_date promised_delivery_date  \
0  ORD00001     CUST006    PROD006  2025-04-16             2025-05-04   
1  ORD00002     CUST002    PROD008  2025-01-23             2025-02-04   
2  ORD00003     CUST001    PROD009  2025-04-29             2025-05-12   
3  ORD00004     CUST015    PROD001  2025-02-22             2025-03-05   
4  ORD00005     CUST016    PROD002  2025-01-08             2025-01-25   

  actual_delivery_date  quantity  unit_price        plant_location  \
0           2025-05-05        37     3018.62  San Jose, Costa Rica   
1           2025-02-04        42      720.95  San Jose, Costa Rica   
2           2025-05-10        25     1989.32      Santa Clara, USA   
3           2025-03-07        17     3346.13  San Jose, Costa Rica   
4           2025-01-27        31      880.72  Alajuela, Costa Rica   

  order_status  
0      Shipped  
1      Shipped  
2      Shipped  
3      Shipped  
4      Shipped  
order_id       

In [ ]:
# 1. duplicated
print(o[o.order_id.duplicated(keep=False)])

# 2. strange values
print(o[(o.quantity <= 0) | o.unit_price.isna() | o.order_date.isna()])

# 3. orphan orders
print("customers without account:", set(o.customer_id) - set(a.customer_id))
print("products without master:", set(o.product_id) - set(p.product_id))

# 4. status vs delivery date
print(pd.crosstab(o.order_status, o.actual_delivery_date.isna()))

# 5. no sense dates
for c in ["order_date", "promised_delivery_date", "actual_delivery_date"]:
    o[c] = pd.to_datetime(o[c], errors="coerce")
print(o[o.actual_delivery_date < o.order_date])

# 6. forecast structure
print(f.groupby(["product_id", "region"]).size().unstack())
print(f.forecast_month.value_counts().sort_index())

     order_id customer_id product_id  order_date promised_delivery_date  \
4    ORD00005     CUST016    PROD002  2025-01-08             2025-01-25   
120  ORD00005     CUST016    PROD002  2025-01-08             2025-01-25   

    actual_delivery_date  quantity  unit_price        plant_location  \
4             2025-01-27        31      880.72  Alajuela, Costa Rica   
120           2025-01-27        99      880.72  Alajuela, Costa Rica   

    order_status  
4        Shipped  
120      Shipped  
     order_id customer_id product_id  order_date promised_delivery_date  \
123  ORD90103     CUST003    PROD008  2025-03-09             2025-03-21   
124  ORD90104     CUST018    PROD007  2025-03-23             2025-04-09   
126  ORD90106     CUST004    PROD008  2025-02-12             2025-03-02   
127  ORD90107     CUST012    PROD006         NaN             2025-04-30   

    actual_delivery_date  quantity  unit_price        plant_location  \
123           2025-03-23       -15     1054.40  San 